In [ ]:

import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# define hyperparameters and constants
EPOCHS = 50          # We set a high number, but EarlyStopping will find the best epoch.
LEARNING_RATE = 0.001
DROPOUT_RATE = 0.5
MODEL_PATH = 'best_fashion_mnist_model.keras' # The name of the file where our trained model will be saved.

# load and preprocess
print("Loading and preprocessing data...")
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

# normalize pixel values from 0-255 to 0-1
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# add a channel dimension for the CNN (from 28x28 to 28x28x1)
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# one-hot encode the labels
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)
print("Data loading complete. 👍")

# define class names for plotting
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# visualize
print("\nDisplaying sample images from the training set...")
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_train[i].reshape(28, 28), cmap='gray')
    plt.xlabel(class_names[y_train[i]])
plt.suptitle("Sample Images from the Dataset", fontsize=16)
plt.show()


# create the cnn model
print("\nCreating the CNN model architecture...")
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(DROPOUT_RATE),
    Dense(10, activation='softmax')
])

# compile the model with our chosen optimizer and loss function
optimizer = Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# display the model's architecture
model.summary()

# data augmentation & callbacks
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)
train_generator = datagen.flow(x_train, y_train_cat, batch_size=64)

# callbacks for smart training
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
# modelCheckpoint to save the model only when validation loss improves
model_checkpoint = ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True)

# --- 6. TRAIN THE MODEL ---
print("\n🚀 Starting model training... (This will take a few minutes)")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=(x_test, y_test_cat),
    callbacks=[early_stopping, model_checkpoint]
)
print(f"\n✅ Training complete! The best model has been saved to '{MODEL_PATH}'.")


# --- 7. VISUALIZE TRAINING HISTORY ---
print("\nVisualizing training history...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Model Training History', fontsize=16)
# Plot training & validation accuracy values
ax1.plot(history.history['accuracy'])
ax1.plot(history.history['val_accuracy'])
ax1.set_title('Model Accuracy')
ax1.set_ylabel('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend(['Train', 'Validation'], loc='upper left')
# Plot training & validation loss values
ax2.plot(history.history['loss'])
ax2.plot(history.history['val_loss'])
ax2.set_title('Model Loss')
ax2.set_ylabel('Loss')
ax2.set_xlabel('Epoch')
ax2.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# --- 8. ✨ NEW: EVALUATE MODEL PERFORMANCE ---
print(f"\nLoading the best model from '{MODEL_PATH}' for evaluation...")
best_model = load_model(MODEL_PATH)

# Make predictions on the test set
y_pred_probs = best_model.predict(x_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = y_test # Use original integer labels

# --- 8a. Classification Report ---
print("\n📊 Classification Report:\n")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_names))

# --- 8b. Confusion Matrix ---
print("\n🔍 Generating Confusion Matrix...")
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=16)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# --- 9. ✨ NEW: VISUALIZE PREDICTIONS ---
print("\nVisualizing model predictions on test images...")
plt.figure(figsize=(12, 12))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')

    predicted_label = y_pred_classes[i]
    true_label = y_true_classes[i]

    # Set color to green for correct prediction, red for incorrect
    color = 'green' if predicted_label == true_label else 'red'

    plt.xlabel(f"Pred: {class_names[predicted_label]}\nTrue: {class_names[true_label]}", color=color)

plt.suptitle("Model Predictions (Green=Correct, Red=Incorrect)", fontsize=16, y=0.93)
plt.show()


# --- 10. DOWNLOAD THE SAVED MODEL FILE ---
# This special Colab command will trigger a download in your browser.
from google.colab import files
print(f"\nPreparing to download '{MODEL_PATH}'...")
files.download(MODEL_PATH)

: 